# Exploration d'un run

Ce notebook **n'exécute aucune simulation** : il lit les artefacts d'un run déjà
produit par le pipeline.

```bash
python main.py run --config experiments/full_grid.yaml
```

Toute la logique d'expérience vit dans `src/pipeline/` et se pilote en ligne de
commande. Ce notebook sert uniquement à explorer les résultats de façon
interactive — y ajouter du code de simulation ou de configuration recréerait la
duplication que le pipeline a supprimée.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

import pandas as pd
from IPython.display import Image, display

from src.pipeline.store import RunStore
from src.pipeline import figures

## Choix du run

`RunStore.latest()` prend le plus récent ; passer un chemin à `RunStore.open()`
pour en cibler un autre.

In [ ]:
for path in RunStore.list_runs('../results_grid'):
    print(path)

In [ ]:
store = RunStore.latest('../results_grid')
params = store.read_params()
manifest = store.read_manifest()

print(store.root)
print(params.describe())
print(f"graine={params.seed} | commit={manifest['git_commit']} | "
      f"cas={manifest['nb_cases_done']}/{manifest['nb_cases_planned']}")

## Tableau de synthèse

Une ligne par cas (scénario x flotte x méthode).

In [ ]:
summary = pd.read_csv(store.summary_path)

colonnes = ['scenario', 'nb_cars', 'method', 'exact_satisfaction',
            'needs_satisfaction', 'mean_travel_distance_km',
            'mean_waiting_time_min', 'total_ms_mean',
            'nb_reservations', 'nb_pres', 'nb_no_show',
            'nb_early_canc', 'nb_late_canc', 'invariant_ok']
summary[colonnes]

## Comparaison des méthodes

Les deux méthodes d'un même couple (scénario, flotte) tournent sur le **même**
monde initial : l'écart mesuré est imputable à la méthode, pas au tirage.

In [ ]:
pivot = summary.pivot_table(
    index=['scenario', 'nb_cars'],
    columns='method',
    values=['exact_satisfaction', 'mean_travel_distance_km', 'total_ms_mean'],
)
pivot

## Figures

Déjà écrites par le pipeline dans `figures/`. Pour les régénérer après avoir
modifié `src/pipeline/figures.py` :

```bash
python main.py report --latest
```

In [ ]:
for path in sorted(store.figures_dir.glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))

## Tables détaillées

Une table par cas : `stations`, `behaviors`, `acceptances`, `alpha`, `latency`.

In [ ]:
from src.pipeline.params import CaseParams

case = CaseParams(scenario=params.scenarios[-1],
                  nb_cars=params.fleet_sizes[-1],
                  method='bramev')

stations = pd.read_csv(store.table_path(case, 'stations'))
stations[['station_id', 'nb_charg_spot', 'alpha', 'occupancy_rate',
          'nb_reservations', 'nb_pres', 'nb_no_show', 'nb_late_canc',
          'nb_offer_issued', 'nb_offer_expired']].head(15)

In [ ]:
latency = pd.read_csv(store.table_path(case, 'latency'))
latency[['demand_id', 'nb_stations', 'nb_offers', 'confirmed',
         'first_offer_ms', 'last_offer_ms', 'selection_ms',
         'confirmation_ms', 'total_ms']].describe()

## Intention tirée vs issue observée

Écart entre les probabilités du scénario et ce qui se réalise effectivement.

In [ ]:
behaviors = pd.read_csv(store.table_path(case, 'behaviors'))
behaviors.pivot_table(index='label', columns='kind', values='count', fill_value=0)

In [ ]:
for result in store.iter_results():
    for warning in result['behaviors'].get('diagnostics', []):
        print(f"[{result['scenario']}/{result['config']['nb_cars']}/"
              f"{result['mode']}] {warning}\n")